In [1]:
!pip install -q fastparquet
!pip install -q statsforecast

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.6/354.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 46.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 2.8 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import TSB, AutoETS, AutoARIMA, Naive, SeasonalNaive, Theta, OptimizedTheta, CrostonOptimized, ADIDA, IMAPA, CrostonSBA, WindowAverage

%matplotlib inline

#####################################################
# Импортируем .py файлы с уже написанными функциями #
#####################################################

import sys
sys.path.append('/kaggle/input/datasets/faibus/diploma')

# функции для расчёта метрик
from metrics import (
    DEFAULT_METRICS,
    get_model_columns,
    compute_metrics_per_window,
    summarize_metrics,
    summarize_metrics_by_segments,
)

# функции для фильтрации и разделения рядов
from split_utils import filter_long_series, split_train_val_test

# функции для оценки и сбора предсказаний
from evaluation_utils import evaluate_frozen_windows

In [3]:
# данные о реальном спросе
real_demand = pd.read_parquet('/kaggle/input/datasets/faibus/diploma/real_demand_data.parquet', engine='fastparquet')

# выгружаем типы рядов
ts_dict_df = pd.read_csv('/kaggle/input/datasets/faibus/diploma/ts_dict_df')[['SKU_id', 'ts_type']]

In [4]:
# Выделяем SKU, принадлежащие к типу lumpy
lumpy_ts = list(ts_dict_df.query(" ts_type == 'intermittent' ")['SKU_id'])

# фильтруем датасет по lumpy_ts
real_demand = real_demand.query(" SKU_id.isin(@lumpy_ts) ")

# причесываем датасет
real_demand = real_demand.rename(columns = {'date':'ds', 'real_demand':'y', 'SKU_id':'unique_id'})[['unique_id', 'ds', 'y']]

real_demand.shape

(2722522, 3)

### Фильтруем ряды

In [5]:
from split_utils import filter_long_series, split_train_val_test

real_demand_filtered = filter_long_series(
    real_demand,
    horizon=14,
    n_val_windows=1,
    n_test_windows=3,
    min_train_points=35,
    id_col="unique_id",
)

eval_df = real_demand_filtered[["unique_id", "ds", "y"]].sort_values(["unique_id", "ds"])

### Обучение

In [7]:
# 1. Создаем список с моделями
models = [
    Naive(alias='Naive'),  
    SeasonalNaive(season_length=7, alias='SeasonalNaive7'),    
    WindowAverage(window_size=7, alias='WindowAvg_7'), 
    OptimizedTheta(season_length=7, alias='OptTheta'),
    AutoETS(alias='AutoETS'),  
    CrostonOptimized(alias='CrostonOpt'),
    CrostonSBA(alias='CrostonSBA'),
    TSB(alpha_d=0.2, alpha_p=0.2, alias='TSB_02_02'),
    ADIDA(alias='ADIDA'),
    IMAPA(alias='IMAPA')                                 
]

# обучаем только на train, val - подбор наилучших параметров, test - итоговое качество метрик
sf = StatsForecast(models=models, freq="D", n_jobs=1)
cv_frozen = sf.cross_validation(
    df=eval_df,
    h=14,
    step_size=14,
    n_windows=4,   # 1 val + 3 test
    refit=False,   # не переобучаем параметры
)

### Собираем прогнозы

In [8]:
val_cutoff = pd.Timestamp("2025-08-05")
test_cutoffs = [
    pd.Timestamp("2025-08-19"),
    pd.Timestamp("2025-09-02"),
    pd.Timestamp("2025-09-16"),
]

val_pred = cv_frozen[cv_frozen["cutoff"] == val_cutoff].copy()
test_pred = cv_frozen[cv_frozen["cutoff"].isin(test_cutoffs)].copy()

cutoff_to_window = {c: i + 1 for i, c in enumerate(test_cutoffs)}
test_pred["test_window"] = test_pred["cutoff"].map(cutoff_to_window)

### Считаем метрики

In [9]:
from metrics import DEFAULT_METRICS, get_model_columns, compute_metrics_per_window, summarize_metrics

val_model_cols = get_model_columns(
    val_pred,
    reserved_columns=("unique_id", "ds", "cutoff", "y", "index", "test_window"),
)
val_metrics_per_window = compute_metrics_per_window(val_pred, val_model_cols, DEFAULT_METRICS)
val_summary_mean, val_summary_stats = summarize_metrics(val_metrics_per_window)

test_model_cols = get_model_columns(
    test_pred,
    reserved_columns=("unique_id", "ds", "cutoff", "y", "index", "test_window"),
)
test_metrics_per_window = compute_metrics_per_window(test_pred, test_model_cols, DEFAULT_METRICS)
test_summary_mean, test_summary_stats = summarize_metrics(test_metrics_per_window)

print("VAL mean metrics:")
display(val_summary_mean)

print('')

print("TEST mean metrics (3 windows x 14 days):")
display(test_summary_mean)

VAL mean metrics:


,mae,rmse,smape,wape
model,,,,
ADIDA,0.468095,0.933741,178.065405,118.199853
AutoETS,0.473842,0.942490,178.060639,119.651004
CrostonOpt,0.539211,1.003485,176.681003,136.157415
CrostonSBA,0.541628,0.978000,176.900300,136.767687
IMAPA,0.464289,0.930296,178.015829,117.238733
Naive,0.513602,1.139972,158.582376,129.690843
OptTheta,0.482311,0.950806,178.370660,121.789528
SeasonalNaive7,0.517108,1.192169,161.174846,130.576256
TSB_02_02,0.462948,0.933262,178.192721,116.900202



TEST mean metrics (3 windows x 14 days):


,mae,rmse,smape,wape
model,,,,
ADIDA,0.457752,0.870903,177.961101,118.725107
AutoETS,0.468405,0.901060,177.986524,121.485571
CrostonOpt,0.514976,0.904196,176.883612,133.561270
CrostonSBA,0.514637,0.901643,177.374279,133.470942
IMAPA,0.455681,0.871657,177.860431,118.190414
Naive,0.512481,1.139914,158.687073,132.956644
OptTheta,0.475445,0.901081,178.037500,123.315367
SeasonalNaive7,0.513974,1.145982,160.264216,133.324318
TSB_02_02,0.457324,0.878973,177.881097,118.620006


In [10]:
test_pred.to_csv("intermittent_forecast.csv")